# Crack Model Comparison on the Sinha Rotor

ROSS provides four transverse crack models, each with a different approach to
representing the breathing mechanism and the resulting stiffness reduction:

| Model | Basis | Breathing law | Max depth ratio |
|---|---|---|---|
| **Mayes** | Linear Fracture Mechanics | Cosine (smooth) | 0.5 |
| **Gasch** | Linear Fracture Mechanics | Fourier-series square wave | 0.5 |
| **Flex Open** | Equivalent beam / 3-D FE | Always open (no breathing) | 0.6 |
| **Flex Breathing** | Equivalent beam / 3-D FE | Stress-based iterative open/close | 0.6 |

This notebook runs all four models on the same Sinha rotor under identical
operating conditions (two speeds, multiple crack depth ratios) and compares:

- Steady-state **orbits** at the probe node
- **Frequency spectra** (FFT) and harmonic content (1X, 2X, 3X)
- **Harmonic amplitudes vs crack depth** for each model
- **2X / 1X diagnostic ratio** vs crack depth
- **Time waveforms** at a fixed crack depth
- **Stiffness variation** over one revolution (breathing function shape)

In [1]:
import numpy as np
import pandas as pd
import ross as rs
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ross.faults.crack import Crack

pio.renderers.default = "notebook"
print(f"ROSS version: {rs.__version__}")

Q_ = rs.Q_

rotor = rs.Rotor.load("sinha_rotor.toml")

ROSS version: 2.2.0


## Configuration

In [20]:
CRACK_MODELS = ["Mayes", "Gasch"] # "Flex Open", "Flex Breathing"]

# Number of divisions around shaft cross-section for breathing crack models
# (affects angular resolution of stiffness variation).
# Used only by the "Mayes" and "Gasch" crack models in ROSS.
CROSS_DIVISIONS = 10

DEPTH_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5]
# PLOT_DEPTH specifies which crack depth ratio (as a fraction of shaft diameter)
# will be used for certain plots and analyses in this notebook.
# It is a convenience parameter: when visualizing orbits, time series, or spectra,
# instead of plotting all depth ratios, we pick this representative value.
# For example, PLOT_DEPTH=0.3 means crack depth = 0.3 × shaft diameter.

PLOT_DEPTH = 0.5

# This parameter affects plotting/analysis for all four crack models in this notebook:
# - Mayes
# - Gasch
# - Flex Open
# - Flex Breathing
# But it does NOT change which crack models are simulated or how the crack breathing/stiffness is computed;
# it simply selects which one of the DEPTH_RATIOS (e.g., 0.1, 0.2, 0.3, 0.4, 0.5) to use for focused illustration,
# regardless of the crack model.

UNB_MAG = Q_(2e-4, "kg*m")
UNB_PHASE = np.deg2rad(315)

SPEED_0 = Q_(650, "rpm")
SPEED_1 = Q_(750, "rpm")
SPEEDS = [SPEED_0, SPEED_1]

DT = 1e-3
T = np.arange(0.0, 2.0, DT)
FREQ_RANGE = Q_((0, 200), "Hz")  # Frequency range for plotting (FRF, spectrum, etc.)

DISK_NODE = rotor.disk_elements[0].n
CRACK_NODE = DISK_NODE + 1
PROBE_NODE = rotor.bearing_elements[1].n - 1

probe = [rs.Probe(node=PROBE_NODE, angle=0.0)]

ndof = rotor.number_dof
nodes = rotor.nodes
# link_nodes: nodes in the rotor model that are "linked" to other nodes across, e.g., rigid couplings or shaft connections.
# Used here to determine if the probe node lies in a linked segment (which affects DOF indexing for that node).
link_nodes = rotor.link_nodes

# Compute DOF indices for the probe node (where we measure response).
# In ROSS, each node (station) has `ndof` degrees of freedom (typically 4 or 6 per node).
# For plain shaft sections, DOFs per node are ordered [x, y, ...]; for linked nodes (e.g., across rigid couplings), DOF indexing is offset
# to account for node merging. The fix_dof parameter compensates for this by subtracting the appropriate offset if the probe node is a link node.
fix_dof = (PROBE_NODE - nodes[-1] - 1) * ndof // 2 if PROBE_NODE in link_nodes else 0  # DOF index correction if probe node is a link node
dofx = ndof * PROBE_NODE - fix_dof  # global DOF index for x-direction at the probe node
dofy = ndof * PROBE_NODE + 1 - fix_dof  # global DOF index for y-direction at the probe node

# steady_start marks the index in T where we consider the transient response to have passed and the system to have reached steady state.
# We typically discard data from the first half of the simulation (transients), and analyze only the second half.
# This index is used to slice arrays (e.g., displacement, velocity, time) to extract steady-state sections for orbit, FFT, and statistics calculations.
steady_start = int(len(T) * 0.5)

MODEL_COLORS = {
    "Mayes": "#1f77b4",
    "Gasch": "#ff7f0e",
    "Flex Open": "#2ca02c",
    "Flex Breathing": "#d62728",
}

print(f"Disk node: {DISK_NODE}")
print(f"Crack node: {CRACK_NODE}")
print(f"Probe node: {PROBE_NODE}")
print(f"DOF x: {dofx}, DOF y: {dofy}")
print(f"Depth ratios: {DEPTH_RATIOS}")
print(f"Speeds: {[f'{s}' for s in SPEEDS]}")

Disk node: 6
Crack node: 7
Probe node: 10
DOF x: 60, DOF y: 61
Depth ratios: [0.1, 0.2, 0.3, 0.4, 0.5]
Speeds: ['650 revolutions_per_minute', '750 revolutions_per_minute']


## Simulation Loop

Run each crack model at every depth ratio and speed combination.
Total: 2 speeds x 4 models x 5 depths = 40 simulations.

In [21]:
results = {}
unb_mag_val = UNB_MAG.to("kg*m").m

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_val = speed.to("rad/s").m
    results[speed_rpm] = {}

    for model in CRACK_MODELS:
        results[speed_rpm][model] = {}
        kwargs = {}
        if model == "Flex Breathing":
            kwargs["cross_divisions"] = CROSS_DIVISIONS

        for dr in DEPTH_RATIOS:
            print(f"Running: {speed_rpm} RPM | {model} | depth = {dr:.0%}...")
            res = rotor.run_crack(
                n=CRACK_NODE,
                depth_ratio=dr,
                crack_model=model,
                node=[DISK_NODE],
                unbalance_magnitude=[unb_mag_val],
                unbalance_phase=[UNB_PHASE],
                speed=speed_val,
                t=T,
                model_reduction={"num_modes": 12},
                **kwargs,
            )
            results[speed_rpm][model][dr] = res

total = sum(
    len(depths) for speeds in results.values() for depths in speeds.values()
)
print(f"\nCompleted {total} simulations.")

Running: 650 RPM | Mayes | depth = 10%...
Running with model reduction: pseudomodal
Running: 650 RPM | Mayes | depth = 20%...
Running with model reduction: pseudomodal
Running: 650 RPM | Mayes | depth = 30%...
Running with model reduction: pseudomodal
Running: 650 RPM | Mayes | depth = 40%...
Running with model reduction: pseudomodal
Running: 650 RPM | Mayes | depth = 50%...
Running with model reduction: pseudomodal
Running: 650 RPM | Gasch | depth = 10%...
Running with model reduction: pseudomodal
Running: 650 RPM | Gasch | depth = 20%...
Running with model reduction: pseudomodal
Running: 650 RPM | Gasch | depth = 30%...
Running with model reduction: pseudomodal
Running: 650 RPM | Gasch | depth = 40%...
Running with model reduction: pseudomodal
Running: 650 RPM | Gasch | depth = 50%...
Running with model reduction: pseudomodal
Running: 750 RPM | Mayes | depth = 10%...
Running with model reduction: pseudomodal
Running: 750 RPM | Mayes | depth = 20%...
Running with model reduction: pseu

## Orbit Comparison

Steady-state orbits at the probe node for each crack model at a fixed depth
ratio. A circular orbit indicates dominant 1X; multi-lobed shapes reveal
higher harmonics introduced by the crack.

In [22]:
for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)

    fig_orbits = make_subplots(
        rows=1, cols=4,
        subplot_titles=CRACK_MODELS,
        horizontal_spacing=0.08,
    )

    for i, model in enumerate(CRACK_MODELS):
        res = results[speed_rpm][model][PLOT_DEPTH]
        x_ss = res.yout[steady_start:, dofx] * 1e6
        y_ss = res.yout[steady_start:, dofy] * 1e6

        fig_orbits.add_trace(
            go.Scatter(
                x=x_ss, y=y_ss,
                mode="lines",
                line=dict(color=MODEL_COLORS[model], width=1),
                showlegend=False,
            ),
            row=1, col=i + 1,
        )
        fig_orbits.update_xaxes(title_text="X (μm)", row=1, col=i + 1)
        fig_orbits.update_yaxes(title_text="Y (μm)", row=1, col=i + 1)

    fig_orbits.update_layout(
        title=dict(text=f"Orbits at Probe Node — depth = {PLOT_DEPTH:.0%} — {speed_rpm} RPM"),
        height=400,
        width=1200,
    )
    fig_orbits.show()

## FFT Spectra Overlay

Overlaid frequency spectra from all four models at the same crack depth.
Vertical dashed lines mark the 1X, 2X and 3X harmonics.

In [5]:
freq_range_hz = FREQ_RANGE.to("Hz").m

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_hz = speed.to("Hz").m

    fig_fft = go.Figure()

    for model in CRACK_MODELS:
        res = results[speed_rpm][model][PLOT_DEPTH]
        x_ss = res.yout[steady_start:, dofx]
        dt = T[1] - T[0]
        N = len(x_ss)
        freqs = np.fft.rfftfreq(N, d=dt)
        fft_amp = 2.0 / N * np.abs(np.fft.rfft(x_ss))

        mask = (freqs >= freq_range_hz[0]) & (freqs <= freq_range_hz[1])

        fig_fft.add_trace(
            go.Scatter(
                x=freqs[mask],
                y=fft_amp[mask] * 1e6,
                mode="lines",
                name=model,
                line=dict(color=MODEL_COLORS[model], width=1.5),
            )
        )

    for nx in [1, 2, 3]:
        fig_fft.add_vline(
            x=nx * speed_hz,
            line=dict(color="gray", width=1, dash="dash"),
            annotation_text=f"{nx}X",
            annotation_position="top right",
        )

    fig_fft.update_layout(
        title=dict(text=f"Frequency Spectra — depth = {PLOT_DEPTH:.0%} — {speed_rpm} RPM"),
        xaxis_title="Frequency (Hz)",
        yaxis_title="Amplitude (μm)",
        height=500,
        width=1000,
        legend=dict(title="Crack Model"),
    )
    fig_fft.show()

## Harmonic Amplitudes vs Crack Depth

Extract the 1X, 2X and 3X harmonic amplitudes from the FFT at each depth
ratio and plot them as a function of crack severity. Each line represents a
different crack model.

In [6]:
def get_harmonic_amplitude(yout, dof, t, target_freq_hz):
    """Extract the FFT amplitude at the frequency bin closest to target_freq_hz."""
    signal = yout[steady_start:, dof]
    dt = t[1] - t[0]
    N = len(signal)
    freqs = np.fft.rfftfreq(N, d=dt)
    fft_vals = 2.0 / N * np.abs(np.fft.rfft(signal))
    idx = np.argmin(np.abs(freqs - target_freq_hz))
    return fft_vals[idx]


harmonics_tables = {}

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_hz = speed.to("Hz").m
    harmonics_tables[speed_rpm] = {}

    for model in CRACK_MODELS:
        rows = {"depth": [], "1X": [], "2X": [], "3X": []}
        for dr in DEPTH_RATIOS:
            res = results[speed_rpm][model][dr]
            rows["depth"].append(dr)
            rows["1X"].append(get_harmonic_amplitude(res.yout, dofx, T, speed_hz) * 1e6)
            rows["2X"].append(get_harmonic_amplitude(res.yout, dofx, T, 2 * speed_hz) * 1e6)
            rows["3X"].append(get_harmonic_amplitude(res.yout, dofx, T, 3 * speed_hz) * 1e6)
        harmonics_tables[speed_rpm][model] = pd.DataFrame(rows)

    print(f"\n=== {speed_rpm} RPM ===")
    for model in CRACK_MODELS:
        print(f"\n--- {model} ---")
        print(harmonics_tables[speed_rpm][model].to_string(index=False, float_format="{:.4f}".format))


=== 650 RPM ===

--- Mayes ---
 depth      1X     2X     3X
0.1000 12.6896 0.2071 0.1340
0.2000 12.7638 0.2915 0.2214
0.3000 12.8826 0.4390 0.3393
0.4000 13.0193 0.6281 0.4404
0.5000 13.1523 0.8426 0.4580

--- Gasch ---
 depth      1X     2X     3X
0.1000 12.6891 0.2294 0.1355
0.2000 12.7637 0.3987 0.2317
0.3000 12.8905 0.6925 0.3728
0.4000 13.0479 1.0700 0.5105
0.5000 13.2098 1.4662 0.5624

=== 750 RPM ===

--- Mayes ---
 depth      1X     2X     3X
0.1000 12.0799 0.5699 0.2317
0.2000 12.1635 0.9151 0.1851
0.3000 12.2996 1.5901 0.1493
0.4000 12.4593 2.5941 0.1455
0.5000 12.6164 3.8681 0.1105

--- Gasch ---
 depth      1X     2X     3X
0.1000 12.0801 0.6649 0.2315
0.2000 12.1681 1.3824 0.1808
0.3000 12.3249 2.7447 0.1372
0.4000 12.5320 4.7424 0.1585
0.5000 12.7567 7.1416 0.1836


In [7]:
for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)

    fig_harmonics = make_subplots(
        rows=1, cols=3,
        subplot_titles=["1X", "2X", "3X"],
        horizontal_spacing=0.08,
    )

    for col_idx, harmonic in enumerate(["1X", "2X", "3X"], start=1):
        for model in CRACK_MODELS:
            df = harmonics_tables[speed_rpm][model]
            fig_harmonics.add_trace(
                go.Scatter(
                    x=[f"{d:.0%}" for d in df["depth"]],
                    y=df[harmonic],
                    mode="lines+markers",
                    name=model,
                    line=dict(color=MODEL_COLORS[model], width=2),
                    marker=dict(size=7),
                    showlegend=(col_idx == 1),
                    legendgroup=model,
                ),
                row=1, col=col_idx,
            )
        fig_harmonics.update_xaxes(title_text="Depth Ratio", row=1, col=col_idx)
        fig_harmonics.update_yaxes(title_text="Amplitude (μm)", row=1, col=col_idx)

    fig_harmonics.update_layout(
        title=dict(text=f"Harmonic Amplitudes vs Crack Depth — {speed_rpm} RPM"),
        height=450,
        width=1200,
        legend=dict(title="Crack Model"),
    )
    fig_harmonics.show()

## 2X / 1X Amplitude Ratio vs Crack Depth

The 2X / 1X ratio is a key diagnostic indicator for breathing cracks. Comparing
how it grows with crack depth across models reveals which models predict
stronger crack signatures at a given severity.

In [8]:
for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)

    fig_ratio = go.Figure()

    for model in CRACK_MODELS:
        df = harmonics_tables[speed_rpm][model]
        ratio_2x_1x = df["2X"] / df["1X"]

        fig_ratio.add_trace(
            go.Scatter(
                x=[f"{d:.0%}" for d in df["depth"]],
                y=ratio_2x_1x,
                mode="lines+markers",
                name=model,
                line=dict(color=MODEL_COLORS[model], width=2),
                marker=dict(size=8),
            )
        )

    fig_ratio.update_layout(
        title=dict(text=f"2X / 1X Ratio vs Crack Depth — {speed_rpm} RPM"),
        xaxis_title="Depth Ratio",
        yaxis_title="2X / 1X Amplitude Ratio",
        height=450,
        width=800,
        legend=dict(title="Crack Model"),
    )
    fig_ratio.show()

## Steady-State Time Waveforms

Compare the time-domain vibration waveform over two shaft revolutions for all
four crack models at the same depth ratio. Waveform distortion (departure from
a pure sine) reflects the harmonic content produced by each model.

In [9]:
n_revs = 2

for speed in SPEEDS:
    speed_rpm = int(speed.to("rpm").m)
    speed_hz = speed.to("Hz").m
    period = 1.0 / speed_hz
    t_window = n_revs * period
    t_start_ss = T[steady_start]
    t_end_window = t_start_ss + t_window
    mask_time = (T >= t_start_ss) & (T <= t_end_window)

    fig_time = go.Figure()

    for model in CRACK_MODELS:
        res = results[speed_rpm][model][PLOT_DEPTH]
        fig_time.add_trace(
            go.Scatter(
                x=(T[mask_time] - t_start_ss) / period,
                y=res.yout[mask_time, dofx] * 1e6,
                mode="lines",
                name=model,
                line=dict(color=MODEL_COLORS[model], width=1.5),
            )
        )

    fig_time.update_layout(
        title=dict(text=f"Time Waveform — depth = {PLOT_DEPTH:.0%} — {speed_rpm} RPM"),
        xaxis_title="Revolutions",
        yaxis_title="Displacement (μm)",
        height=450,
        width=1000,
        legend=dict(title="Crack Model"),
    )
    fig_time.show()

## Stiffness Variation Over One Revolution

Each crack model prescribes a different breathing law — the way the cracked
element's stiffness varies as the shaft rotates through one full revolution.
This plot shows the effective bending stiffness K(0,0) of the cracked element
as a function of angular position for Mayes, Gasch and Flex Open.

The Flex Breathing model is excluded here because its stiffness depends on
the instantaneous stress field (displacement response), so it cannot be
evaluated purely as a function of angular position.

In [10]:
angles = np.linspace(0, 2 * np.pi, 360)
stiffness_models = ["Mayes", "Gasch"] #, "Flex Open"]

fig_stiff = go.Figure()

for model_name in stiffness_models:
    crack_obj = Crack(rotor, n=CRACK_NODE, depth_ratio=PLOT_DEPTH, crack_model=model_name)
    model_func = crack_obj._crack_model

    k_vals = np.array([model_func(ap)[0, 0] for ap in angles])
    k_intact = crack_obj.K_elem[0, 0]
    k_normalized = k_vals / k_intact

    fig_stiff.add_trace(
        go.Scatter(
            x=np.degrees(angles),
            y=k_normalized,
            mode="lines",
            name=model_name,
            line=dict(color=MODEL_COLORS[model_name], width=2),
        )
    )

fig_stiff.update_layout(
    title=dict(text=f"Normalized Bending Stiffness Over One Revolution — depth = {PLOT_DEPTH:.0%}"),
    xaxis_title="Angular Position (°)",
    yaxis_title="K / K_intact",
    xaxis=dict(tickvals=[0, 90, 180, 270, 360]),
    height=450,
    width=900,
    legend=dict(title="Crack Model"),
)
fig_stiff.show()

## Observations

1. **Breathing law shape**: Mayes uses a smooth cosine to transition between
   open and closed states, while Gasch approximates a square wave via Fourier
   series. Flex Open keeps the crack permanently open (constant reduced
   stiffness regardless of angular position).

2. **Orbit distortion**: At the same crack depth, the Flex Open model produces
   the most distorted orbits because the stiffness asymmetry is constant and
   never "heals" during rotation. The breathing models (Mayes, Gasch, Flex
   Breathing) yield orbits whose shape depends on the breathing duty cycle.

3. **2X harmonic**: Breathing models generate a strong 2X component because the
   stiffness switches twice per revolution — this is the classical crack
   signature. The Flex Open model, with no breathing, generates 2X content
   purely from the rotating stiffness asymmetry and typically produces
   different relative amplitudes.

4. **Depth sensitivity**: All models show the 2X / 1X ratio growing with crack
   depth, but the growth rate differs. This has practical implications: a
   detection threshold calibrated on one model may not transfer directly to
   another.

5. **Model choice guidance**:
   - **Mayes / Gasch** are appropriate when the crack is known to breathe under
     gravity-dominated bending. Gasch's sharper transition can capture higher
     harmonics (3X, 4X) more strongly.
   - **Flex Open** is suitable for horizontal rotors where centrifugal forces
     keep the crack open, or for worst-case stiffness reduction analysis.
   - **Flex Breathing** is the most physically realistic for rotors where the
     local stress field governs crack opening, but it is computationally more
     expensive due to the iterative cross-section integration.